In [ ]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"     # use PyTorch's sampler, no compilation needed
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # engine runs inside the notebook process

In [ ]:
%pip install datasets

In [ ]:
import sys; print(sys.executable)

In [ ]:
from huggingface_hub import HfApi, whoami

try:
    print("Logged in as:", whoami()["name"])
except Exception as e:
    print("NOT logged in:", e)

print([d.id for d in HfApi().list_datasets(author="Maci10")])

In [ ]:
from huggingface_hub import login
login()   # a box appears; paste the token there (never in the code itself)

In [ ]:
math500.push_to_hub("Maci10/dpo-king-math500", split="train", private=True)

In [ ]:
from datasets import load_dataset
math500 = load_dataset('Maci10/dpo-king-math500', split = 'test')
print(len(math500))

In [ ]:
sft_all = load_dataset('Maci10/dpo-king-sft_all', split = 'train')
math_val = load_dataset('Maci10/dpo-king-math_val', split = 'train')

print(len(sft_all), len(math_val))

In [ ]:
import sys; sys.path.insert(0, ".")
from vllm import LLM, SamplingParams
from posttrain.prompts import build_prompt
from posttrain.grader import mv_grade
from posttrain.data import load_split
from datasets import load_dataset

math500 = load_dataset('Maci10/dpo-king-math500', split = 'test')

llm = LLM(
    model = "Qwen/Qwen3-0.6B-Base",
    dtype = "bfloat16",
    max_model_len = 2048,
    gpu_memory_utilization = 0.85,
    seed = 0
)


# Stop on token IDs: <|im_end|> is a special token, and vLLM strips special
# tokens from the output text, so a stop *string* can miss it.
tok = llm.get_tokenizer()
stop_ids = [tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")]
params = SamplingParams(temperature=0.0, max_tokens = 1024, stop_token_ids = stop_ids,)

batch = math500.select(range(5))
prompts = [build_prompt(ex['problem']) for ex in batch]
outputs = llm.generate(prompts, params)

for ex, out in zip(batch, outputs):
    text = out.outputs[0].text
    print("GOLD:", ex["answer"], "| CORRECT:", mv_grade(ex["answer"], text),
          "| tokens:", len(out.outputs[0].token_ids))
    print(text[-300:], "\n" + "-" * 60)

# 03 — Evaluation on MATH-500 (reused for every arm)

- so in the code above we tested the base model on a few random samples from the math500 test set now we are going to evaluate it on the whole test set

- few shot prompting = giving AI examples before asking it to do a task
- zero shot prompting = just giving the model the question and having it generate you the answer

Change the **config cell** (`MODEL_ID`, `ARM`) and rerun for each arm. Everything else stays identical,
so every arm is evaluated with exactly the same prompts, sampling settings and grader.

**Arm 0 (base model)** is run two ways:
- **zero-shot, chat format**: the same prompt every trained arm gets (the starting point)
- **few-shot, plain text**: 4 worked MATH-train examples, the fair way to measure what a *base* model knows

Trained arms (SFT, DPO, GRPO) only need the zero-shot run.

## 1. Config — the only cell to change between arms

In [ ]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"     # image's nvcc is CUDA 12.x -> FlashInfer JIT fails
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # run the engine inside the notebook process

%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, ".")
import json, torch, vllm
from datasets import load_dataset
from vllm import LLM, SamplingParams

from posttrain.prompts import (build_prompt, stop_token_ids, select_fewshot_examples,
                               build_fewshot_prompt, FEWSHOT_STOP_STRINGS)
from posttrain.evaluate import run_generation, grade, summarize, print_summary, save_run
from posttrain import grader

## 1. Config — the only cell to change between arms

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B-Base"   # later: path or Hub ID of the SFT / DPO / GRPO checkpoint
ARM = "arm0_base"                   # later: "armA_sft", "armB_dpo", ...
RUN_FEWSHOT = True                  # only for the base model

HF_RESULTS_REPO = "Maci10/dpo-king-eval-results"   # private dataset repo, one config per run
MAX_TOKENS = 1024
SEED = 0

GPU_NAME = torch.cuda.get_device_name(0)
META = {"model": MODEL_ID, "arm": ARM, "gpu": GPU_NAME, "vllm": vllm.__version__,
        "max_tokens": MAX_TOKENS, "temperature": 0.0, "seed": SEED}
print(META)

## 2. Data + grader check

In [ ]:
math500 = load_dataset("Maci10/dpo-king-math500", split = "test")

print(len(math500), math500.column_names)

## 3. Load the model — run this cell ONCE per kernel session

In [ ]:
llm = LLM(
    model = MODEL_ID,
    dtype = "bfloat16",
    max_model_len = 4096,
    gpu_memory_utilization = 0.85,
    enforce_eager = True,
    seed = SEED,
)

STOP_IDS = stop_token_ids(llm.get_tokenizer())
print("stop ids:", STOP_IDS)

- model = MODEL_ID : downloads  (first time) and loads the weights onto the GPU
- dtype = 'bfloat16' : the number format Qwen3 was trained in: half the memory of full precison with no meaningful accuray loss
- max_model_len = 4096 : the longest sequence (prompt + answer) the enginer can handle 4,096 leaves room for the long few-shot prompt plus 1,024 genreated tokens 

- gpu_memory_utilization = 0.85 : vLLm reserves 85% of GPU memory up front; about 1.2GB for the weights and the rest for KV_Cache

- stop tokens: vLLM does not remove special tokens from what the model generates, it still produces <|im_end|> when it is done but vLLM will hide that token when it turns the generated token IDs back into text.

- Stopping on token IDs checks the raw tokens as they’re generated, before any conversion to text, so it always works. stop_token_ids() just looks up the ID numbers for <|im_end|> (151645) and <|endoftext|> (151643). Your printout should show those two numbers.

In [ ]:
print(STOP_IDS)